In [ ]:
import sys
sys.path.append('/home/james/ml-proj/predmain/src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import joblib
from sklearn.model_selection import train_test_split
from sklearn.metrics import matthews_corrcoef

from features import engineer_features, get_feature_columns, get_target_column
from models import build_model, get_class_weight, tune_model
from evaluate import evaluate_model, plot_confusion_matrix

sns.set_theme(style='darkgrid')

# Load and engineer features
df = pd.read_csv('/home/james/ml-proj/predmain/data/ai4i2020.csv')
df = df.drop(columns=['UDI', 'Product ID', 'Type'], errors='ignore')
df = engineer_features(df)

X = df[get_feature_columns()]
y = df[get_target_column()]

# 80/20 stratified split (stratify preserves the 3.4% failure rate in both sets)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

weight = get_class_weight(y_train)
print(f"Features: {get_feature_columns()}")
print(f"Train: {len(X_train)} | Test: {len(X_test)}")
print(f"Split: {len(X_train)/len(df)*100:.0f}% train / {len(X_test)/len(df)*100:.0f}% test")

In [ ]:
# Q: Can Optuna find better hyperparameters than our hand-picked defaults?
# (TPE sampler, 50 trials, optimising MCC via 5-fold stratified CV)
tune_results = tune_model(X_train, y_train, scale_pos_weight=weight, n_trials=50)

print(f"\nDefault model MCC baseline to beat:")
default_cv = evaluate_model(build_model(scale_pos_weight=weight), X_train, y_train)

In [ ]:
# Build tuned model, fit on full training set, and evaluate on held-out test
model = build_model(scale_pos_weight=weight, **tune_results['best_params'])
model.fit(X_train, y_train)

# Q: Does the tuned model beat the Naive Bayes baseline of MCC 0.235?
results = evaluate_model(model, X_train, y_train)
print(f"\nImprovement over NB baseline: {results['mcc_mean'] - 0.235:+.3f} MCC")

In [ ]:
# Q: Where is the model failing — false alarms or missed failures?
test_metrics = plot_confusion_matrix(
    model, X_test, y_test,
    save_path='/home/james/ml-proj/predmain/outputs/figures/confusion_matrix.png',
)

In [ ]:
# Q: Which engineered features is the model actually using?
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

# Bar plot — overall importance
shap.summary_plot(shap_values, X_test, plot_type='bar', show=False)
plt.title('Q: Which features matter most overall?')
plt.tight_layout()
plt.savefig('/home/james/ml-proj/predmain/outputs/figures/shap_importance.png', dpi=150)
plt.show()

# Dot plot — direction and magnitude
shap.summary_plot(shap_values, X_test, show=False)
plt.title('Q: At what values does each feature push toward failure?\n(Red = high feature value, Blue = low)')
plt.tight_layout()
plt.savefig('/home/james/ml-proj/predmain/outputs/figures/shap_detail.png', dpi=150)
plt.show()

In [ ]:
# Q: Did feature engineering actually improve on raw features?
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import make_scorer

# Raw features baseline
df_raw = pd.read_csv('/home/james/ml-proj/predmain/data/ai4i2020.csv')
df_raw = df_raw.drop(columns=['UDI', 'Product ID', 'Type'], errors='ignore')
raw_features = ['Air temperature [K]', 'Process temperature [K]', 
                'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]']
X_raw = df_raw[raw_features]
y_raw = df_raw['Machine failure']

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
mcc_scorer = make_scorer(matthews_corrcoef)

weight_full = get_class_weight(y)
model_raw = build_model(scale_pos_weight=weight_full)
model_eng = build_model(scale_pos_weight=weight_full, **tune_results['best_params'])

mcc_raw = cross_val_score(model_raw, X_raw, y_raw, cv=skf, scoring=mcc_scorer)
mcc_eng = cross_val_score(model_eng, X, y, cv=skf, scoring=mcc_scorer)
f1_raw  = cross_val_score(model_raw, X_raw, y_raw, cv=skf, scoring='f1')
f1_eng  = cross_val_score(model_eng, X, y, cv=skf, scoring='f1')

fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(2)
width = 0.3

ax.bar(x - width/2, [mcc_raw.mean(), f1_raw.mean()], width, 
       label='Raw features (5)', color='lightcoral',
       yerr=[mcc_raw.std(), f1_raw.std()], capsize=5)
ax.bar(x + width/2, [mcc_eng.mean(), f1_eng.mean()], width, 
       label='Engineered + tuned (3)', color='steelblue',
       yerr=[mcc_eng.std(), f1_eng.std()], capsize=5)

ax.set_xticks(x)
ax.set_xticklabels(['MCC', 'F1 Score'])
ax.set_ylabel('Score (higher = better)')
ax.set_title('Q: Did feature engineering + tuning improve the model?')
ax.legend()
ax.set_ylim(0, 1)

for i, (r, e) in enumerate(zip([mcc_raw.mean(), f1_raw.mean()], 
                                 [mcc_eng.mean(), f1_eng.mean()])):
    ax.text(i - width/2, r + 0.02, f'{r:.3f}', ha='center', fontsize=9)
    ax.text(i + width/2, e + 0.02, f'{e:.3f}', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('/home/james/ml-proj/predmain/outputs/figures/raw_vs_engineered.png', dpi=150)
plt.show()

print(f"MCC improvement: {mcc_eng.mean() - mcc_raw.mean():+.4f}")
print(f"F1  improvement: {f1_eng.mean()  - f1_raw.mean():+.4f}")

In [ ]:
# Save tuned model
MODEL_PATH = '/home/james/ml-proj/predmain/outputs/models/xgb_model.pkl'
joblib.dump(model, MODEL_PATH)
print(f"Tuned model saved to {MODEL_PATH}")

# Summary
print(f"\n{'='*50}")
print(f"Tuned XGBoost — Test set metrics:")
print(f"  MCC: {test_metrics['mcc']:.3f}")
print(f"  F1:  {test_metrics['f1']:.3f}")
print(f"  Best Optuna params: {tune_results['best_params']}")
print(f"{'='*50}")